# ColSmol ViDoRe Encoder

Encode ViDoRe corpus pages using ColSmol and export sharded `.pkl` index for retrieval notebooks.

Output format per shard:
- `fused_index`: list of token embeddings (`np.ndarray`, variable length)
- `page_keys`: list of unique page keys
- `metadata`: list of metadata dicts (contains `domain`, `corpus_id`, `join_doc_name`, `safe_page`, etc.)

In [ ]:
deps_path = '/kaggle/input/datasets/nhhsag12/colpali-dependency'
!pip install --no-index --find-links {deps_path} --requirement {deps_path}/requirements.txt

In [ ]:
import os
import io
import gc
import glob
import json
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
from tqdm.notebook import tqdm

from colpali_engine.models import ColIdefics3, ColIdefics3Processor
from peft import PeftModel

In [ ]:
# ================================================================
# CONFIG
# ================================================================
VIDORE_DATASET_ROOT = "/kaggle/input/datasets/namthi/vidore-v3"
DOMAIN_FILTER = []  # [] means all domains

COLSMOL_BASE = "/kaggle/input/models/nhhsag12/colsmolvlm-instruct-500m-base/pytorch/default/1"
COLSMOL_LORA = "/kaggle/input/models/nhhsag12/colsmol-500m/pytorch/default/4"

OUTPUT_DIR = "/kaggle/working/vidore_colsmol_encoded_shards"
MANIFEST_NAME = "vidore_colsmol_manifest.pkl"

BATCH_SIZE = 16
SHARD_SIZE = 2000  # pages per shard
SAVE_DTYPE = "float16"  # float16 (smaller/faster IO) or float32
OVERWRITE_OUTPUT = False

assert SAVE_DTYPE in {"float16", "float32"}

print("CONFIG READY")
print(f"VIDORE_DATASET_ROOT: {VIDORE_DATASET_ROOT}")
print(f"OUTPUT_DIR        : {OUTPUT_DIR}")
print(f"BATCH_SIZE        : {BATCH_SIZE}")
print(f"SHARD_SIZE        : {SHARD_SIZE}")
print(f"SAVE_DTYPE        : {SAVE_DTYPE}")

In [ ]:
# ================================================================
# RTX Pro 6000 optimization profile
# ================================================================
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for fast encoding.")

device = "cuda:0"

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

gpu_name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU: {gpu_name}")
print(f"VRAM: {total_gb:.2f} GB")
print("TF32 + cuDNN benchmark enabled")

In [ ]:
# ================================================================
# Load ColSmol encoder
# ================================================================
base_model = ColIdefics3.from_pretrained(
    COLSMOL_BASE,
    torch_dtype=torch.bfloat16,
    device_map=device,
    attn_implementation="eager"
)
model = PeftModel.from_pretrained(base_model, COLSMOL_LORA).eval()
processor = ColIdefics3Processor.from_pretrained(COLSMOL_LORA)

print("ColSmol model loaded.")

In [ ]:
# ================================================================
# Data helpers
# ================================================================
def norm_key(v):
    s = "" if pd.isna(v) else str(v).strip()
    if not s:
        return ""
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except Exception:
        pass
    return s


def choose_row_value(row, keys, default=None):
    for k in keys:
        if k in row and not pd.isna(row[k]):
            return row[k]
    return default


def to_pil_image(value, domain_dir):
    if isinstance(value, Image.Image):
        return value.convert("RGB")

    if isinstance(value, dict):
        if value.get("bytes") is not None:
            return Image.open(io.BytesIO(value["bytes"])).convert("RGB")
        if value.get("path"):
            p = str(value["path"])
            cand = [p, str(domain_dir / p)]
            for cp in cand:
                if os.path.isfile(cp):
                    return Image.open(cp).convert("RGB")

    if isinstance(value, (bytes, bytearray)):
        return Image.open(io.BytesIO(value)).convert("RGB")

    if isinstance(value, str):
        p = value
        cand = [p, str(domain_dir / p)]
        for cp in cand:
            if os.path.isfile(cp):
                return Image.open(cp).convert("RGB")

    return None


def extract_pil_from_row(row_dict, domain_dir):
    image_cols = ["image", "page_image", "pil_image", "image_bytes", "image_path", "path"]
    for c in image_cols:
        if c in row_dict:
            try:
                img = to_pil_image(row_dict[c], domain_dir)
                if img is not None:
                    return img
            except Exception:
                continue
    return None


def build_meta(row_dict, domain_name, fallback_idx):
    corpus_id = norm_key(choose_row_value(row_dict, ["corpus_id", "id", "doc_id", "page_id"], default=fallback_idx))
    page_no = choose_row_value(row_dict, ["page", "page_idx", "page_no", "page_number"], default=0)
    try:
        safe_page = int(page_no)
    except Exception:
        safe_page = 0

    join_doc_name = choose_row_value(
        row_dict,
        ["join_doc_name", "doc_name", "document_id", "pdf_name", "source", "filename"],
        default=f"{domain_name}_{corpus_id}"
    )
    join_doc_name = str(join_doc_name)

    return {
        "domain": str(domain_name),
        "corpus_id": str(corpus_id),
        "safe_page": int(safe_page),
        "join_doc_name": join_doc_name,
    }


def iter_vidore_corpus_rows(dataset_root, domain_filter):
    root = Path(dataset_root)
    if not root.exists():
        raise FileNotFoundError(f"Dataset root not found: {dataset_root}")

    domain_dirs = [p for p in sorted(root.iterdir()) if p.is_dir()]
    if domain_filter:
        domain_dirs = [p for p in domain_dirs if p.name in set(domain_filter)]

    for domain_dir in domain_dirs:
        domain_name = domain_dir.name
        corpus_files = []
        for fp in sorted(domain_dir.rglob("*.parquet")):
            pstr = str(fp).replace('\\', '/').lower()
            if '/corpus/' in pstr:
                corpus_files.append(fp)

        for parquet_fp in corpus_files:
            try:
                df = pd.read_parquet(parquet_fp)
            except Exception as ex:
                print(f"[WARN] Skip unreadable parquet: {parquet_fp} | {ex}")
                continue

            if df is None or df.empty:
                continue

            for i in range(len(df)):
                row_dict = df.iloc[i].to_dict()
                img = extract_pil_from_row(row_dict, domain_dir)
                if img is None:
                    continue

                meta = build_meta(row_dict, domain_name, fallback_idx=i)
                meta["source_parquet"] = str(parquet_fp)
                yield img, meta

In [ ]:
# ================================================================
# Progress estimator helper
# ================================================================
def estimate_total_pages(dataset_root, domain_filter):
    root = Path(dataset_root)
    if not root.exists():
        raise FileNotFoundError(f"Dataset root not found: {dataset_root}")

    domain_dirs = [p for p in sorted(root.iterdir()) if p.is_dir()]
    if domain_filter:
        domain_dirs = [p for p in domain_dirs if p.name in set(domain_filter)]

    total_rows = 0
    for domain_dir in domain_dirs:
        for fp in sorted(domain_dir.rglob("*.parquet")):
            pstr = str(fp).replace('\\', '/').lower()
            if '/corpus/' not in pstr:
                continue
            try:
                total_rows += len(pd.read_parquet(fp, columns=[]))
            except Exception:
                # Fallback if parquet engine does not allow empty-column read
                try:
                    total_rows += len(pd.read_parquet(fp))
                except Exception:
                    continue
    return int(total_rows)


def sec_to_hms(seconds):
    seconds = max(float(seconds), 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

In [ ]:
# ================================================================
# Encoding helpers
# ================================================================
def encode_image_batch(pil_images):
    inputs = processor.process_images(pil_images).to(device)

    with torch.inference_mode():
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            out = model(**inputs)

    if isinstance(out, tuple):
        out = out[0]

    attention_mask = inputs.get("attention_mask", None)
    embs = []
    for i in range(out.shape[0]):
        token_emb = out[i]
        if attention_mask is not None:
            keep = torch.where(attention_mask[i] > 0)[0]
            if keep.numel() > 0:
                token_emb = token_emb[keep]

        arr = token_emb.detach().float().cpu().numpy()
        if SAVE_DTYPE == "float16":
            arr = arr.astype(np.float16)
        else:
            arr = arr.astype(np.float32)
        embs.append(arr)

    return embs


def flush_shard(output_dir, shard_idx, shard_embeddings, shard_page_keys, shard_metadata):
    shard_path = os.path.join(output_dir, f"shard_{shard_idx:05d}.pkl")
    payload = {
        "fused_index": shard_embeddings,
        "page_keys": shard_page_keys,
        "metadata": shard_metadata,
    }
    with open(shard_path, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
    return shard_path

In [ ]:
# ================================================================
# Run encoding and export sharded PKL
# ================================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)

existing_shards = sorted(glob.glob(os.path.join(OUTPUT_DIR, "shard_*.pkl")))
if existing_shards and not OVERWRITE_OUTPUT:
    raise RuntimeError(
        f"Output already has {len(existing_shards)} shard files. Set OVERWRITE_OUTPUT=True or use a new OUTPUT_DIR."
    )

if OVERWRITE_OUTPUT and existing_shards:
    for fp in existing_shards:
        os.remove(fp)

total_pages = 0
total_skipped = 0
shard_idx = 0

batch_images = []
batch_meta = []

shard_embeddings = []
shard_page_keys = []
shard_metadata = []
shard_files = []

t0 = time.perf_counter()

estimated_total_pages = estimate_total_pages(VIDORE_DATASET_ROOT, DOMAIN_FILTER)
if estimated_total_pages <= 0:
    estimated_total_pages = None

row_iter = iter_vidore_corpus_rows(VIDORE_DATASET_ROOT, DOMAIN_FILTER)
pbar = tqdm(row_iter, total=estimated_total_pages, desc="Encoding pages", unit="page")
if estimated_total_pages is not None:
    print(f"Estimated corpus pages: {estimated_total_pages}")

for img, meta in pbar:
    batch_images.append(img)
    batch_meta.append(meta)

    if len(batch_images) < BATCH_SIZE:
        continue

    try:
        embs = encode_image_batch(batch_images)
    except Exception as ex:
        total_skipped += len(batch_images)
        print(f"[WARN] Skip batch of {len(batch_images)} due to encode error: {ex}")
        batch_images.clear()
        batch_meta.clear()
        continue

    for emb, meta_i in zip(embs, batch_meta):
        page_key = f"{meta_i['domain']}::{meta_i['corpus_id']}::{meta_i['safe_page']}"
        shard_embeddings.append(emb)
        shard_page_keys.append(page_key)
        shard_metadata.append(meta_i)

    total_pages += len(embs)
    batch_images.clear()
    batch_meta.clear()

    elapsed_now = time.perf_counter() - t0
    speed = (total_pages / elapsed_now) if elapsed_now > 0 else 0.0
    if estimated_total_pages is not None:
        remaining_pages = max(estimated_total_pages - total_pages, 0)
        eta_sec = (remaining_pages / speed) if speed > 0 else 0.0
        pbar.set_postfix({
            'done': total_pages,
            'remain': remaining_pages,
            'eta': sec_to_hms(eta_sec),
            'p/s': f"{speed:.2f}"
        })
    else:
        pbar.set_postfix({
            'done': total_pages,
            'eta': 'unknown',
            'p/s': f"{speed:.2f}"
        })

    if len(shard_embeddings) >= SHARD_SIZE:
        out_fp = flush_shard(OUTPUT_DIR, shard_idx, shard_embeddings, shard_page_keys, shard_metadata)
        shard_files.append(out_fp)
        shard_idx += 1
        shard_embeddings, shard_page_keys, shard_metadata = [], [], []
        gc.collect()
        torch.cuda.empty_cache()

# Flush last partial batch
if batch_images:
    try:
        embs = encode_image_batch(batch_images)
        for emb, meta_i in zip(embs, batch_meta):
            page_key = f"{meta_i['domain']}::{meta_i['corpus_id']}::{meta_i['safe_page']}"
            shard_embeddings.append(emb)
            shard_page_keys.append(page_key)
            shard_metadata.append(meta_i)
        total_pages += len(embs)
    except Exception as ex:
        total_skipped += len(batch_images)
        print(f"[WARN] Skip final batch of {len(batch_images)} due to encode error: {ex}")

# Flush last shard
if shard_embeddings:
    out_fp = flush_shard(OUTPUT_DIR, shard_idx, shard_embeddings, shard_page_keys, shard_metadata)
    shard_files.append(out_fp)

elapsed = time.perf_counter() - t0
pps = (total_pages / elapsed) if elapsed > 0 else 0.0

manifest = {
    "format": "vidore_sharded_v1",
    "model_name": "colsmol",
    "base_model": COLSMOL_BASE,
    "lora_model": COLSMOL_LORA,
    "dataset_root": VIDORE_DATASET_ROOT,
    "domain_filter": DOMAIN_FILTER,
    "dtype": SAVE_DTYPE,
    "batch_size": BATCH_SIZE,
    "shard_size": SHARD_SIZE,
    "num_pages": total_pages,
    "num_skipped": total_skipped,
    "num_shards": len(shard_files),
    "elapsed_sec": elapsed,
    "pages_per_sec": pps,
    "shards": shard_files,
}

manifest_fp = os.path.join(OUTPUT_DIR, MANIFEST_NAME)
with open(manifest_fp, "wb") as f:
    pickle.dump(manifest, f, protocol=pickle.HIGHEST_PROTOCOL)

manifest_json_fp = os.path.join(OUTPUT_DIR, MANIFEST_NAME.replace(".pkl", ".json"))
with open(manifest_json_fp, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("\nEncoding completed.")
print(f"Total pages encoded : {total_pages}")
print(f"Total pages skipped : {total_skipped}")
print(f"Total shards        : {len(shard_files)}")
print(f"Elapsed (sec)       : {elapsed:.2f}")
print(f"Throughput (p/s)    : {pps:.2f}")
print(f"Manifest PKL        : {manifest_fp}")
print(f"Manifest JSON       : {manifest_json_fp}")

In [ ]:
# Quick sanity check: inspect first shard
first_shards = sorted(glob.glob(os.path.join(OUTPUT_DIR, "shard_*.pkl")))
if not first_shards:
    raise RuntimeError("No shard files found after encoding.")

with open(first_shards[0], "rb") as f:
    sample = pickle.load(f)

print(f"First shard: {first_shards[0]}")
print(f"Entries    : {len(sample.get('fused_index', []))}")
if sample.get('fused_index'):
    e0 = sample['fused_index'][0]
    print(f"Embedding[0] shape: {e0.shape}, dtype: {e0.dtype}")
print(f"Metadata keys: {list(sample.get('metadata', [{}])[0].keys()) if sample.get('metadata') else []}")